In [1]:
import os

import boto3
from dotenv import load_dotenv

In [2]:
load_dotenv()

S3_BUCKET = os.getenv("S3_BUCKET")

STORAGE_OPTIONS={
        "endpoint_url": os.getenv("MLFLOW_S3_ENDPOINT_URL"), 
        "key": os.getenv("AWS_ACCESS_KEY_ID"), 
        "secret": os.getenv("AWS_SECRET_ACCESS_KEY"),
        #"client_kwargs":{"region_name": os.getenv("AWS_REGION")},
        #"config_kwargs": {"signature_version": "s3v4"}
    }

In [3]:
# Проверка содержимого


In [4]:
# Удаление всех версий файлов в S3
def delete_all_obj(bucket_name, endpoint_url, key, secret):
    s3 = boto3.resource('s3',
        endpoint_url=endpoint_url,
        aws_access_key_id=key,
        aws_secret_access_key=secret)
        
    bucket = s3.Bucket(bucket_name)

    # Deleting all versions (works for non-versioned buckets too).
    bucket.object_versions.delete()

    # Aborting all multipart uploads, which also deletes all parts.
    for multipart_upload in bucket.multipart_uploads.iterator():
        # Part uploads that are currently in progress may or may not succeed,
        # so it might be necessary to abort a multipart upload multiple times.
        while len(list(multipart_upload.parts.all())) > 0:
            multipart_upload.abort()

In [5]:
# Очистка S3
delete_all_obj(S3_BUCKET, **STORAGE_OPTIONS)